In [1]:
#imports
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

In [2]:
#Load Data

# Load training and test data
train_df = pd.read_parquet('cleaned_train.parquet')
test_df = pd.read_parquet('cleaned_test.parquet')

# Load Week 9 saved files
with open('tfidf_vectorizer.pkl', 'rb') as f:
    tfidf_vectorizer = pickle.load(f)

with open('item_to_tfidf.pkl', 'rb') as f:
    item_to_tfidf = pickle.load(f)

with open('items_with_text.pkl', 'rb') as f:
    items_with_text = pickle.load(f)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Items with TF-IDF vectors: {len(item_to_tfidf)}")

Train shape: (26580, 3)
Test shape: (6645, 3)
Items with TF-IDF vectors: 391


In [3]:
#Create User Profiles

def create_user_profile(user_id):
    """Create user profile by averaging TF-IDF vectors of items they rated"""
    
    # Get items this user rated
    user_items = train_df[train_df['user_id'] == user_id]['item_id'].tolist()
    
    # Collect TF-IDF vectors for these items
    vectors = []
    for item in user_items:
        if item in item_to_tfidf:
            vec = item_to_tfidf[item]
            # Convert sparse matrix to dense if needed
            if hasattr(vec, 'toarray'):
                vec = vec.toarray()[0]
            vectors.append(vec)
    
    if vectors:
        # Average all vectors
        return np.mean(vectors, axis=0)
    else:
        return None

# Create profiles for all users
print("Creating user profiles...")
user_profiles = {}
users = train_df['user_id'].unique()

for i, user in enumerate(users):
    if i % 200 == 0:
        print(f"  Processed {i}/{len(users)} users")
    user_profiles[user] = create_user_profile(user)

profiles_count = sum(1 for p in user_profiles.values() if p is not None)
print(f"\nUsers with profiles: {profiles_count} out of {len(users)}")

Creating user profiles...
  Processed 0/1389 users
  Processed 200/1389 users
  Processed 400/1389 users
  Processed 600/1389 users
  Processed 800/1389 users
  Processed 1000/1389 users
  Processed 1200/1389 users

Users with profiles: 1386 out of 1389


In [4]:
#Create Content-Based Recommendations

def get_content_recommendations(user_id, n=10):
    """Generate top-n recommendations based on content similarity"""
    
    # Get user profile
    user_profile = user_profiles.get(user_id)
    if user_profile is None:
        return []
    
    # Get items user already rated
    rated_items = set(train_df[train_df['user_id'] == user_id]['item_id'].tolist())
    
    # Calculate similarity to all items
    similarities = []
    for item_id, item_vec in item_to_tfidf.items():
        if item_id not in rated_items:
            # Convert to dense if needed
            if hasattr(item_vec, 'toarray'):
                item_vec = item_vec.toarray()[0]
            
            # Cosine similarity
            sim = cosine_similarity([user_profile], [item_vec])[0][0]
            similarities.append((item_id, sim))
    
    # Sort and return top n
    similarities.sort(key=lambda x: x[1], reverse=True)
    return [item for item, _ in similarities[:n]]

# Test on a sample user
sample_user = users[0]
sample_recs = get_content_recommendations(sample_user, n=10)
print(f"Sample user: {sample_user}")
print(f"Top 5 recommendations: {sample_recs[:5]}")

Sample user: AHOGCWGRSFQ6YZH6QLYUMNQ4N3KA
Top 5 recommendations: ['B07YBXFF99', 'B00NQREKHS', 'B00KCCNMYW', 'B0095CZL8A', 'B06XBN6NCH']


In [5]:
# Generate recommendations for all test users

# Get test users
test_users = test_df['user_id'].unique()
print(f"\nGenerating recommendations for {len(test_users)} test users...")

# Generate recommendations
user_recs = {}
for i, user in enumerate(test_users):
    if i % 100 == 0:
        print(f"  Processed {i}/{len(test_users)} users")
    user_recs[user] = get_content_recommendations(user, n=10)

print(f"\nDone! Generated recommendations for {len(user_recs)} users")


Generating recommendations for 1389 test users...
  Processed 0/1389 users
  Processed 100/1389 users
  Processed 200/1389 users
  Processed 300/1389 users
  Processed 400/1389 users
  Processed 500/1389 users
  Processed 600/1389 users
  Processed 700/1389 users
  Processed 800/1389 users
  Processed 900/1389 users
  Processed 1000/1389 users
  Processed 1100/1389 users
  Processed 1200/1389 users
  Processed 1300/1389 users

Done! Generated recommendations for 1389 users


In [7]:
#Define Evaluation Metrics

def get_relevant_items(user_id, threshold=3):
    """Get items user rated highly in test set"""
    user_test = test_df[test_df['user_id'] == user_id]
    return set(user_test[user_test['rating'] >= threshold]['item_id'].tolist())

def calculate_hit_rate(recs, relevant):
    """1 if at least one relevant item in recommendations"""
    if len(relevant) == 0:
        return None
    return 1 if len(set(recs) & relevant) > 0 else 0

def calculate_precision(recs, relevant, k=10):
    """Precision@k = (# relevant in top-k) / k"""
    if len(relevant) == 0:
        return None
    return len(set(recs[:k]) & relevant) / k

def calculate_ap(recs, relevant, k=10):
    """Average Precision@k"""
    if len(relevant) == 0:
        return None
    ap = 0
    num_relevant = 0
    for i, item in enumerate(recs[:k], 1):
        if item in relevant:
            num_relevant += 1
            ap += num_relevant / i
    return ap / min(k, len(relevant))

def calculate_mrr(recs, relevant, k=10):
    """Mean Reciprocal Rank@k"""
    if len(relevant) == 0:
        return None
    for i, item in enumerate(recs[:k], 1):
        if item in relevant:
            return 1 / i
    return 0

def calculate_coverage(recs_dict, all_items):
    """Coverage = (# unique items recommended) / (total items)"""
    recommended = set()
    for recs in recs_dict.values():
        recommended.update(recs)
    return len(recommended) / len(all_items)

print("\nDefined evaluation metrics")


Defined evaluation metrics


In [8]:
#Calculate Metrics

print("Calculating metrics...")

hit_rates = []
precisions = []
aps = []
mrrs = []

for user in test_users:
    recs = user_recs.get(user, [])
    if not recs:
        continue
    
    relevant = get_relevant_items(user)
    if len(relevant) == 0:
        continue
    
    hr = calculate_hit_rate(recs, relevant)
    if hr is not None:
        hit_rates.append(hr)
    
    p = calculate_precision(recs, relevant)
    if p is not None:
        precisions.append(p)
    
    ap = calculate_ap(recs, relevant)
    if ap is not None:
        aps.append(ap)
    
    mrr = calculate_mrr(recs, relevant)
    if mrr is not None:
        mrrs.append(mrr)

# Calculate coverage
all_items = set(train_df['item_id'].unique()) | set(test_df['item_id'].unique())
coverage = calculate_coverage(user_recs, all_items)

print("\n" + "="*50)
print("CONTENT-BASED RECOMMENDER RESULTS")
print("="*50)
print(f"Hit Rate: {np.mean(hit_rates):.4f}")
print(f"Precision@10: {np.mean(precisions):.4f}")
print(f"MAP@10: {np.mean(aps):.4f}")
print(f"MRR@10: {np.mean(mrrs):.4f}")
print(f"Coverage: {coverage:.4f}")

Calculating metrics...

CONTENT-BASED RECOMMENDER RESULTS
Hit Rate: 0.1273
Precision@10: 0.0143
MAP@10: 0.0114
MRR@10: 0.0426
Coverage: 0.4024


In [9]:
#Save Results

# Create results table
week10_results = pd.DataFrame([{
    'Model': 'Content-Based (TF-IDF)',
    'Hit Rate': np.mean(hit_rates),
    'Precision@10': np.mean(precisions),
    'MAP@10': np.mean(aps),
    'MRR@10': np.mean(mrrs),
    'Coverage': coverage
}])

week10_results.to_csv('week10_results.csv', index=False)
print("\nResults saved to week10_results.csv")
print(week10_results.to_string(index=False))


Results saved to week10_results.csv
                 Model  Hit Rate  Precision@10  MAP@10   MRR@10  Coverage
Content-Based (TF-IDF)  0.127273      0.014255 0.01137 0.042605  0.402361


In [10]:
#Save user profiles and recommendations for future use

with open('user_profiles.pkl', 'wb') as f:
    pickle.dump(user_profiles, f)

with open('user_recs_content.pkl', 'wb') as f:
    pickle.dump(user_recs, f)

print("\nSaved files for Week 11:")
print("  - user_profiles.pkl")
print("  - user_recs_content.pkl")



Saved files for Week 11:
  - user_profiles.pkl
  - user_recs_content.pkl
